## Extracting Past data using yahoo finance

In [3]:
import yfinance as yf
import pandas as pd
# Pull HDFC Bank data (use .NS for NSE, .BO for BSE)
ticker = yf.Ticker('HDFCBANK.NS')
# The .T transposes so dates are rows, metrics are columns
income_stmt   = ticker.financials.T
balance_sheet = ticker.balance_sheet.T
cash_flow     = ticker.cashflow.T
# Save raw files — never modify these
income_stmt.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\income_statement.csv')
balance_sheet.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\balance_sheet.csv')
cash_flow.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\cash_flow.csv')
# Print available columns to understand what data you have
print('Income Statement columns:')
print(income_stmt.columns.tolist())
print(f'Years available: {income_stmt.index.tolist()}')

Income Statement columns:
['Tax Effect Of Unusual Items', 'Tax Rate For Calcs', 'Total Unusual Items', 'Total Unusual Items Excluding Goodwill', 'Net Income From Continuing Operation Net Minority Interest', 'Reconciled Depreciation', 'Net Interest Income', 'Interest Expense', 'Interest Income', 'Normalized Income', 'Net Income From Continuing And Discontinued Operation', 'Diluted Average Shares', 'Basic Average Shares', 'Diluted EPS', 'Basic EPS', 'Diluted NI Availto Com Stockholders', 'Net Income Common Stockholders', 'Net Income', 'Minority Interests', 'Net Income Including Noncontrolling Interests', 'Net Income Extraordinary', 'Net Income Continuous Operations', 'Tax Provision', 'Pretax Income', 'Other Non Operating Income Expenses', 'Special Income Charges', 'Gain On Sale Of Security', 'Depreciation Amortization Depletion Income Statement', 'Depreciation And Amortization In Income Statement', 'Amortization', 'Amortization Of Intangibles Income Statement', 'Depreciation Income State

### Since yfinance gives only 4 years of data and its not enough to train ML model for forecasting I switched to Screener for Data extraction 
Another benefit of doing so is Screener gives Data in RBI's recommeded format which is relevant for HDFC as it's listed in India 

In [5]:
import pandas as pd
import os

# Create the directory
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Balance Sheet
balance_sheet = None
for i, table in enumerate(tables):
    if "Share Capital" in table.iloc[:, 0].values:
        balance_sheet = table
        break

# Export the 10-year historical data to CSV
if balance_sheet is not None:
    # Set the first column (line items) as the index before saving
    balance_sheet.set_index(balance_sheet.columns[0], inplace=True)
    balance_sheet.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_balance_sheet.csv')
    print("Success! 10-year Balance Sheet saved to data/raw/screener_balance_sheet.csv.")
else:
    print("Error: Could not locate the Balance Sheet table.")

Error: Could not locate the Balance Sheet table.


In [8]:
import pandas as pd
import os

# Create the directory
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Balance Sheet using Bank-specific line items
balance_sheet = None
for i, table in enumerate(tables):
    # Convert the first column to strings to check for keywords safely
    first_col = table.iloc[:, 0].astype(str)
    
    # Banks list "Deposits" and "Capital" instead of generic "Share Capital"
    if first_col.str.contains('Deposits', case=False, na=False).any() or \
       first_col.str.contains('Capital', case=False, na=False).any():
        balance_sheet = table
        print(f"✅ Balance sheet found at table index {i}!")
        break

# Export the 10-year historical data to CSV (TRANSPOSED)
if balance_sheet is not None:
    # 1. Set the first column (line items) as the index
    balance_sheet.set_index(balance_sheet.columns[0], inplace=True)
    
    # 2. Transpose the dataframe so Dates become rows and Line Items become columns
    balance_sheet_transposed = balance_sheet.T
    
    # 3. Export to CSV
    balance_sheet_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_balance_sheet.csv')
    print("Success! Transposed 10-year Balance Sheet saved to data/raw/screener_balance_sheet.csv.")
else:
    print("❌ Error: Could not locate the Balance Sheet table.")

✅ Balance sheet found at table index 6!
Success! Transposed 10-year Balance Sheet saved to data/raw/screener_balance_sheet.csv.


In [9]:
import pandas as pd
import os

# Create the directory just in case
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Cash Flow Statement
cash_flow = None
for i, table in enumerate(tables):
    first_col = table.iloc[:, 0].astype(str)
    
    # Looking for standard Cash Flow line items
    if first_col.str.contains('Operating Activity', case=False, na=False).any() or \
       first_col.str.contains('Net Cash Flow', case=False, na=False).any():
        cash_flow = table
        print(f"✅ Cash Flow statement found at table index {i}!")
        break

# Export the historical data to CSV (TRANSPOSED)
if cash_flow is not None:
    # 1. Set the line items as the index
    cash_flow.set_index(cash_flow.columns[0], inplace=True)
    
    # 2. Transpose so Dates = rows, Line Items = columns
    cash_flow_transposed = cash_flow.T
    
    # 3. Export to CSV
    cash_flow_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_cash_flow.csv')
    print("Success! Transposed 10-year Cash Flow saved to data/raw/screener_cash_flow.csv.")
else:
    print("❌ Error: Could not locate the Cash Flow table.")

✅ Cash Flow statement found at table index 7!
Success! Transposed 10-year Cash Flow saved to data/raw/screener_cash_flow.csv.


In [10]:
import pandas as pd
import os

# Ensure directory exists
#os.makedirs('data/raw', exist_ok=True)

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Annual Income Statement (Profit & Loss)
income_stmt = None
for i, table in enumerate(tables):
    first_col = table.iloc[:, 0].astype(str)
    
    # 1. Look for 'Net Profit' or 'Revenue'
    has_net_profit = first_col.str.contains('Net Profit', case=False, na=False).any()
    has_revenue = first_col.str.contains('Revenue', case=False, na=False).any()
    
    # 2. Look for 'TTM' in the column headers to ensure it's the Annual table, not Quarterly
    has_ttm_column = any('TTM' in str(col).upper() for col in table.columns)
    
    if (has_net_profit or has_revenue) and has_ttm_column:
        income_stmt = table
        print(f"✅ Annual Income Statement found at table index {i}!")
        break

# Export the historical data to CSV (TRANSPOSED)
if income_stmt is not None:
    # 1. Set the line items as the index
    income_stmt.set_index(income_stmt.columns[0], inplace=True)
    
    # 2. Transpose so Dates = rows, Line Items = columns
    income_stmt_transposed = income_stmt.T
    
    # 3. Export to CSV
    income_stmt_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_income_stmt.csv')
    print("Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.")
else:
    print("❌ Error: Could not locate the Annual Income Statement table.")

❌ Error: Could not locate the Annual Income Statement table.


In [ ]:
import pandas as pd
import os

url = "https://www.screener.in/company/HDFCBANK/consolidated/"
tables = pd.read_html(url)

# Locate and extract the Annual Income Statement (Profit & Loss)
income_stmt = None
for i, table in enumerate(tables):
    first_col = table.iloc[:, 0].astype(str)
    
    #looking for 'net profit' or 'interest earned' (bank specific)
    is_pl_table = first_col.str.contains('Net Profit', case=False, na=False).any() or \
                  first_col.str.contains('Interest Earned', case=False, na=False).any()
    
    #check column count to differentiate annual (10+ cols) from quarterly (~6 cols)
    is_annual_table = len(table.columns) > 8
    
    if is_pl_table and is_annual_table:
        income_stmt = table
        print(f"✅ Annual Income Statement found at table index {i}!")
        break

# Export the historical data to CSV (TRANSPOSED)
if income_stmt is not None:
    # 1. Set the line items as the index
    income_stmt.set_index(income_stmt.columns[0], inplace=True)
    
    # 2. Transpose so Dates = rows, Line Items = columns
    income_stmt_transposed = income_stmt.T
    
    # 3. Export to CSV
    income_stmt_transposed.to_csv('D:\\Financial KPI Dashboard\\data\\raw\\screener_income_stmt.csv')
    print("Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.")
else:
    print("❌ Error: Could not locate the Annual Income Statement table.")
    print("Let's look at what tables we DO have:")
    for i, t in enumerate(tables):
        print(f"\n--- Table {i} --- (Columns: {len(t.columns)})")
        print(t.iloc[:3, 0].tolist())

✅ Annual Income Statement found at table index 0!
Success! Transposed 10-year Income Statement saved to data/raw/screener_income_stmt.csv.
